# Health Intelligence Engine v2 - walkthrough

Synthetic data only. Experimental research prototype, not a medical device.
Run the cells top to bottom (kernel working directory = project root).

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
from engine.generator import generate_dataset
from engine.validator import TelemetryValidator
from engine.health_engine import HealthEngine

data = generate_dataset(["ASTRO_01"])
raw = data.reset_index(drop=True)
print(raw.shape); raw.head()

## 1. Validation (no imputation)

In [ ]:
v = TelemetryValidator().validate(raw)
print(TelemetryValidator().quality_report(v))
v.loc[v.is_corrupted_detected == 1, ["timestamp", "corruption_reason"]].head(12)

## 2. Personal baseline = 24 h rhythm + sleep effect (fitted on the first 48 h)

In [ ]:
engine = HealthEngine().fit(raw)
m = engine.profile["metrics"]["heart_rate_bpm"]
print({k: m[k] for k in ["mean", "std", "raw_std", "circadian_amplitude", "circadian_peak_hour", "sleep_effect", "variance_explained"]})
print("sigma shrank by x", round(m["raw_std"] / m["std"], 1))

## 3. Analysis: events, sleep, report

In [ ]:
result = engine.analyze(raw)
rep = result["report"]
print(rep["status"], rep["anomaly_score"], "-", rep["reason"][:140])
pd.DataFrame([{k: e[k] for k in ["event_id", "type", "severity", "start", "duration_minutes", "peak_zscore", "needs_human_review"]}
              for e in result["events"] if e["needs_human_review"]])

In [ ]:
result["sleep"]["status"], [(n["night_start"][:10], n["hours"], n["short_night"]) for n in result["sleep"]["nights"]]

## 4. Baseline vs observed

In [ ]:
df = result["df"]
fig, ax = plt.subplots(figsize=(14, 4))
sigma = engine.profile["metrics"]["heart_rate_bpm"]["std"]
ax.fill_between(df.timestamp, df.heart_rate_bpm_expected - 2*sigma, df.heart_rate_bpm_expected + 2*sigma, alpha=.2, label="baseline +/- 2 sigma")
ax.plot(df.timestamp, df.heart_rate_bpm, lw=.6, color="k", label="observed")
f = df[df.hybrid_anomaly == 1]
ax.scatter(f.timestamp, f.heart_rate_bpm, s=8, color="r", label="flagged")
ax.legend(); ax.set_ylabel("bpm"); plt.show()

## 5. Real per-reading streaming (what /simulate/next does)

In [ ]:
session = engine.start_stream(raw.iloc[:776])   # history = everything before the first reading we push
for i in range(776, 790):
    t = session.push(raw.iloc[i].to_dict())
    print(t["step_index"], t["intelligence"]["max_zscore"], t["trigger_alert"], t["intelligence"]["explanation"][:80])

## 6. Evaluation (blind astronauts) - see outputs/evaluation_report.md

In [ ]:
print(open("outputs/evaluation_report.md", encoding="utf-8").read()[:3500])